In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd

raw_file_path = (
    "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/"
    "01_Data/raw/datarawFahrraddiebstahl_raw_2026-08-10.csv"
)

df_raw = pd.read_csv(
    raw_file_path,
    encoding="latin1",
    dtype={"LOR": "string"}
)

# Making a working copy that we can clean safely
df = df_raw.copy()

print("Raw dataset:", df_raw.shape)
print("Working copy:", df.shape)

df.head()

Raw dataset: (26965, 11)
Working copy: (26965, 11)


,ANGELEGT_AM,TATZEIT_ANFANG_DATUM,TATZEIT_ANFANG_STUNDE,TATZEIT_ENDE_DATUM,TATZEIT_ENDE_STUNDE,LOR,SCHADENSHOEHE,VERSUCH,ART_DES_FAHRRADS,DELIKT,ERFASSUNGSGRUND
0,08.08.2026,08.08.2026,0,08.08.2026,0,01200521,0,Ja,Rennrad,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern
1,08.08.2026,01.08.2026,11,01.08.2026,13,03601347,2150,Nein,Citybike,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern
2,08.08.2026,08.08.2026,0,08.08.2026,0,01200521,0,Ja,Rennrad,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern
3,08.08.2026,05.08.2026,11,05.08.2026,17,03701554,2650,Nein,Herrenfahrrad,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern
4,08.08.2026,08.08.2026,12,08.08.2026,17,11401034,1499,Nein,Rennrad,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern


In [3]:
# renaming the German column names into clear English names
df = df.rename(columns={
    "ANGELEGT_AM": "record_created_date",
    "TATZEIT_ANFANG_DATUM": "offence_start_date",
    "TATZEIT_ANFANG_STUNDE": "offence_start_hour",
    "TATZEIT_ENDE_DATUM": "offence_end_date",
    "TATZEIT_ENDE_STUNDE": "offence_end_hour",
    "LOR": "lor",
    "SCHADENSHOEHE": "reported_damage_eur",
    "VERSUCH": "attempt",
    "ART_DES_FAHRRADS": "bicycle_type",
    "DELIKT": "offence_category",
    "ERFASSUNGSGRUND": "recording_reason"
})

print("Renamed columns:")
print(df.columns.tolist())

Renamed columns:
['record_created_date', 'offence_start_date', 'offence_start_hour', 'offence_end_date', 'offence_end_hour', 'lor', 'reported_damage_eur', 'attempt', 'bicycle_type', 'offence_category', 'recording_reason']


In [4]:
# converting the three date columns from text into proper dates
date_columns = [
    "record_created_date",
    "offence_start_date",
    "offence_end_date"
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        format="%d.%m.%Y",
        errors="coerce"
    )

print("Invalid dates:")
print(df[date_columns].isna().sum())

print("\nDate range:")
print("Earliest:", df["offence_start_date"].min())
print("Latest:", df["offence_start_date"].max())

Invalid dates:
record_created_date    0
offence_start_date     0
offence_end_date       0
dtype: int64

Date range:
Earliest: 2025-01-01 00:00:00
Latest: 2026-08-08 00:00:00


In [5]:
# combining each date and hour into complete start and end timestamps
df["offence_start"] = (
    df["offence_start_date"]
    + pd.to_timedelta(df["offence_start_hour"], unit="h")
)

df["offence_end"] = (
    df["offence_end_date"]
    + pd.to_timedelta(df["offence_end_hour"], unit="h")
)

df["offence_window_hours"] = (
    df["offence_end"] - df["offence_start"]
).dt.total_seconds() / 3600

print("Negative windows:", (df["offence_window_hours"] < 0).sum())
print("Minimum window:", df["offence_window_hours"].min())
print("Maximum window:", df["offence_window_hours"].max())

df[
    ["offence_start", "offence_end", "offence_window_hours"]
].head()

Negative windows: 0
Minimum window: 0.0
Maximum window: 72.0


,offence_start,offence_end,offence_window_hours
0,2026-08-08 00:00:00,2026-08-08 00:00:00,0.0
1,2026-08-01 11:00:00,2026-08-01 13:00:00,2.0
2,2026-08-08 00:00:00,2026-08-08 00:00:00,0.0
3,2026-08-05 11:00:00,2026-08-05 17:00:00,6.0
4,2026-08-08 12:00:00,2026-08-08 17:00:00,5.0


In [6]:
# creating calendar columns for the time analysis. These are based on the reported offence-start date.
df["year"] = df["offence_start_date"].dt.year
df["month_number"] = df["offence_start_date"].dt.month
df["month_name"] = df["offence_start_date"].dt.month_name()
df["weekday_number"] = df["offence_start_date"].dt.dayofweek
df["weekday_name"] = df["offence_start_date"].dt.day_name()
df["is_weekend"] = df["weekday_number"].isin([5, 6])

df[
    [
        "offence_start_date",
        "year",
        "month_number",
        "month_name",
        "weekday_name",
        "is_weekend"
    ]
].head()

,offence_start_date,year,month_number,month_name,weekday_name,is_weekend
0,2026-08-08,2026,8,August,Saturday,True
1,2026-08-01,2026,8,August,Saturday,True
2,2026-08-08,2026,8,August,Saturday,True
3,2026-08-05,2026,8,August,Wednesday,False
4,2026-08-08,2026,8,August,Saturday,True


In [7]:
# creating useful flags without deleting any records (These flags let us filter cases later without removing anything from the dataset)
df["is_attempt"] = df["attempt"].eq("Ja")
df["is_attempt_unknown"] = df["attempt"].eq("Unbekannt")
df["is_zero_damage"] = df["reported_damage_eur"].eq(0)

df["is_cellar_burglary"] = df["offence_category"].eq(
    "Keller- und Bodeneinbruch"
)

df["is_cellar_location"] = df["recording_reason"].str.contains(
    r"Keller|Boden",
    case=False,
    na=False
)

print("Attempted incidents:", df["is_attempt"].sum())
print("Unknown attempt status:", df["is_attempt_unknown"].sum())
print("Zero-damage records:", df["is_zero_damage"].sum())
print("Cellar burglary cases:", df["is_cellar_burglary"].sum())
print("Any cellar-location cases:", df["is_cellar_location"].sum())

Attempted incidents: 154
Unknown attempt status: 7
Zero-damage records: 234
Cellar burglary cases: 1351
Any cellar-location cases: 1421


In [8]:
# Marking repeated copies after the first matching row
df["is_duplicate_copy"] = df_raw.duplicated(keep="first")

print("Duplicate-looking copies:", df["is_duplicate_copy"].sum())
print("Current rows and columns:", df.shape)

Duplicate-looking copies: 37
Current rows and columns: (26965, 26)


In [9]:
# creating a flag for the fair 2025–2026 comparison. Since 2026 only runs through 8 August, we will compare 1 January–8 August in both years. This flag will make it easy to create a fair year-to-date comparison in Python, SQL and Tableau.
df["is_comparable_ytd"] = (
    (
        df["offence_start_date"].between(
            "2025-01-01", "2025-08-08"
        )
    )
    |
    (
        df["offence_start_date"].between(
            "2026-01-01", "2026-08-08"
        )
    )
)

comparable_counts = (
    df[df["is_comparable_ytd"]]
    .groupby("year")
    .size()
)

print("Comparable period: 1 January–8 August")
print(comparable_counts)

Comparable period: 1 January–8 August
year
2025    11185
2026     9629
dtype: int64


In [10]:
# final quality check
quality_checks = pd.Series({
    "row_count_preserved": len(df) == len(df_raw),
    "no_missing_dates": df[
        ["record_created_date", "offence_start_date", "offence_end_date"]
    ].notna().all().all(),
    "all_lor_codes_have_8_characters": df["lor"].str.len().eq(8).all(),
    "hours_are_between_0_and_23": (
        df["offence_start_hour"].between(0, 23).all()
        and df["offence_end_hour"].between(0, 23).all()
    ),
    "no_negative_damage": df["reported_damage_eur"].ge(0).all(),
    "no_negative_time_windows": df["offence_window_hours"].ge(0).all(),
    "duplicate_copies_match_audit": df["is_duplicate_copy"].sum() == 37
})

display(quality_checks)

print("\nAll checks passed:", quality_checks.all())
print("Final shape:", df.shape)

,0
row_count_preserved,True
no_missing_dates,True
all_lor_codes_have_8_characters,True
hours_are_between_0_and_23,True
no_negative_damage,True
no_negative_time_windows,True
duplicate_copies_match_audit,True



All checks passed: True
Final shape: (26965, 27)


In [11]:
# now exporting the cleaned dataset to the shared processed folder
cleaned_file_path = (
    "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/"
    "01_Data/processed/cleaned_v1.csv"
)

df.to_csv(
    cleaned_file_path,
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned dataset saved successfully:")
print(cleaned_file_path)
print("Exported rows and columns:", df.shape)

Cleaned dataset saved successfully:
/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/01_Data/processed/cleaned_v1.csv
Exported rows and columns: (26965, 27)


In [12]:
# after the cleaned file was saved, I now verify that it can be reopened correctly
df_check = pd.read_csv(
    cleaned_file_path,
    encoding="utf-8-sig",
    dtype={"lor": "string"}
)

print("Reopened shape:", df_check.shape)
print("Same number of rows:", len(df_check) == len(df))
print("Same columns:", df_check.columns.tolist() == df.columns.tolist())
print("All LOR codes still have 8 characters:", df_check["lor"].str.len().eq(8).all())

Reopened shape: (26965, 27)
Same number of rows: True
Same columns: True
All LOR codes still have 8 characters: True


In [13]:
# checking the folder
import os

processed_folder = (
    "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/"
    "01_Data/processed"
)

print("Processed folder exists:", os.path.exists(processed_folder))
print("Files inside the folder:")
print(os.listdir(processed_folder))


Processed folder exists: True
Files inside the folder:
['cleaned_v1.csv', 'cleaned_v2.csv', 'analysis_ready_v2.csv']


In [14]:
import os

cleaned_file_path = (
    "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/"
    "01_Data/processed/cleaned_v1.csv"
)

df.to_csv(
    cleaned_file_path,
    index=False,
    encoding="utf-8-sig"
)

# Ask Drive to finish writing the file
os.sync()

print("File exists:", os.path.exists(cleaned_file_path))
print("Files in processed folder:", os.listdir(processed_folder))

File exists: True
Files in processed folder: ['cleaned_v1.csv', 'cleaned_v2.csv', 'analysis_ready_v2.csv']


In [15]:
df["incident_status"] = df["attempt"].map({
    "Ja": "Attempted",
    "Nein": "Completed",
    "Unbekannt": "Unknown"
})

print(df["incident_status"].value_counts())

incident_status
Completed    26804
Attempted      154
Unknown          7
Name: count, dtype: int64


In [16]:
df_final_check = pd.read_csv(
    cleaned_file_path,
    encoding="utf-8-sig",
    dtype={"lor": "string"}
)

print("Final file shape:", df_final_check.shape)
print("Has incident_status:", "incident_status" in df_final_check.columns)
print("All LOR codes have 8 characters:", df_final_check["lor"].str.len().eq(8).all())

Final file shape: (26965, 27)
Has incident_status: False
All LOR codes have 8 characters: True


In [17]:
import os

# Add the missing category
df["incident_status"] = df["attempt"].map({
    "Ja": "Attempted",
    "Nein": "Completed",
    "Unbekannt": "Unknown"
})

# Save as a new version
cleaned_v2_path = (
    "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/"
    "01_Data/processed/cleaned_v2.csv"
)

df.to_csv(
    cleaned_v2_path,
    index=False,
    encoding="utf-8-sig"
)

os.sync()

print("File created:", os.path.exists(cleaned_v2_path))
print("Dataset shape:", df.shape)
print(df["incident_status"].value_counts())

File created: True
Dataset shape: (26965, 28)
incident_status
Completed    26804
Attempted      154
Unknown          7
Name: count, dtype: int64
